# Investment Research Agent — Final Report

USD AAI-520 Final Team Project.

GitHub: seanmcgowanx/multi-agent-financial-analyst

## 1. Setup

In [1]:
from multi_agent.team import ResearchTeam

ImportError: cannot import name 'SPECIALISTS' from 'multi_agent.specialists' (/Users/seanmcgowan/multi-agent-financial-analyst/multi_agent/specialists/__init__.py)

## 2. Agent Design and Workflows

Architecture diagram, agent roster (name, role, model), how the lab pattern was extended.

## 3. Agent Functions

### 3.1 Planning

In [ ]:
# coordinator.plan(...) for a ticker; show the plan

### 3.2 Dynamic tool use

In [ ]:
# a specialist's use_tools trace: tool choices + observations

### 3.3 Self-reflection

In [ ]:
# evaluator score + feedback on a draft

### 3.4 Learning across runs

In [ ]:
# run twice; show lessons injected into the second plan

## 4. Workflow Patterns

### 4.1 Prompt Chaining

In [ ]:
# NewsChain(...).run(ticker): show each step's output

### 4.2 Routing

In [ ]:
# plan steps -> which specialist handled each

### 4.3 Evaluator-Optimizer

In [ ]:
# drafts per round, scores; chart score vs round

## 5. End-to-End Run

In [ ]:
# team.run(ticker); print final report; team.show_team_status()

## 5.1 Three-Panel Research Dashboard

The panels show daily closing prices, recent quarterly performance, and news volume by publication date. Each panel labels the dates and data source; the news legend names the publishers represented.

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv

load_dotenv()

# Change this ticker to view another company.
ticker = "AAPL"
period = "6mo"
news_days = 7

stock = yf.Ticker(ticker)
try:
    info = stock.get_info() or {}
except Exception:
    info = {}
currency = info.get("financialCurrency") or info.get("currency") or "unspecified"

# Panel 1: daily closing prices.
try:
    price_history = stock.history(period=period, auto_adjust=False)
    close = price_history["Close"].dropna() if not price_history.empty else pd.Series(dtype=float)
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
except Exception:
    close = pd.Series(dtype=float)

# Panel 2: latest four quarterly income statements.
try:
    income = stock.get_income_stmt(freq="quarterly")
except Exception:
    income = pd.DataFrame()
if not income.empty:
    quarter_columns = income.sort_index(axis=1, ascending=False).columns[:4]
    quarterly = income.reindex(
        ["TotalRevenue", "NetIncome", "OperatingIncome"]
    ).loc[:, quarter_columns].T.sort_index()
    quarterly.index = pd.to_datetime(quarterly.index)
else:
    quarterly = pd.DataFrame()

# Panel 3: dated articles, preferring NewsAPI and falling back to Yahoo Finance.
news_api_key = os.getenv("NEWSAPI_KEY")
news_cutoff = datetime.now(timezone.utc) - timedelta(days=news_days)
news_articles = []
news_provider = "NewsAPI"
news_note = None
if news_api_key:
    try:
        response = requests.get(
            "https://newsapi.org/v2/everything",
            params={
                "q": ticker,
                "from": news_cutoff.date().isoformat(),
                "sortBy": "publishedAt",
                "pageSize": 100,
                "language": "en",
            },
            headers={"X-Api-Key": news_api_key},
            timeout=20,
        )
        response.raise_for_status()
        payload = response.json()
        if payload.get("status") != "ok":
            raise ValueError("NewsAPI returned an error")
        news_articles = payload.get("articles", [])
    except (requests.RequestException, ValueError, TypeError):
        news_note = "NewsAPI unavailable; using Yahoo Finance fallback"

if not news_articles:
    news_provider = "Yahoo Finance"
    try:
        news_articles = stock.news or []
    except Exception:
        news_articles = []

news_rows = []
for article in news_articles:
    content = article.get("content") or article
    published = article.get("publishedAt") or content.get("pubDate")
    timestamp = article.get("providerPublishTime")
    if published:
        published_at = pd.to_datetime(published, utc=True, errors="coerce")
    elif isinstance(timestamp, (int, float)):
        published_at = pd.to_datetime(timestamp, unit="s", utc=True, errors="coerce")
    else:
        continue
    if pd.isna(published_at) or published_at.to_pydatetime() < news_cutoff:
        continue

    source = article.get("source") or content.get("provider") or {}
    if isinstance(source, dict):
        publisher = source.get("name") or source.get("displayName")
    else:
        publisher = source
    publisher = publisher or article.get("publisher") or "Unknown publisher"
    news_rows.append({"published_at": published_at, "publisher": publisher})

news = pd.DataFrame(news_rows)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle(
    f"{ticker} Research Snapshot | Retrieved {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}",
    fontsize=15,
)

# Price panel.
price_ax = axes[0]
price_ax.set_title(f"Daily close, {period}\nSource: Yahoo Finance; unadjusted Close")
price_ax.set_xlabel("Trading date")
price_ax.set_ylabel(f"Closing price ({currency})")
price_ax.grid(True, alpha=0.25)
if not close.empty:
    price_ax.plot(close.index, close.values, color="#176b87", linewidth=1.8)
    price_ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=6))
    price_ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
else:
    price_ax.text(0.5, 0.5, "Price data unavailable", ha="center", va="center", transform=price_ax.transAxes)

# Quarterly financials panel: grouped bars plus an operating-margin line.
financial_ax = axes[1]
financial_ax.set_title("Quarterly results, latest 4 periods\nSource: Yahoo Finance income statement")
financial_ax.set_xlabel("Quarter ended")
financial_ax.set_ylabel(f"Amount ({currency}, billions)")
financial_ax.grid(True, axis="y", alpha=0.25)
if not quarterly.empty:
    positions = range(len(quarterly))
    revenue = pd.to_numeric(quarterly["TotalRevenue"], errors="coerce")
    net_income = pd.to_numeric(quarterly["NetIncome"], errors="coerce")
    operating_income = pd.to_numeric(quarterly["OperatingIncome"], errors="coerce")
    financial_ax.bar(
        [position - 0.18 for position in positions],
        revenue / 1_000_000_000,
        width=0.36,
        label="Revenue",
        color="#2a9d8f",
    )
    financial_ax.bar(
        [position + 0.18 for position in positions],
        net_income / 1_000_000_000,
        width=0.36,
        label="Net income",
        color="#e9c46a",
    )
    financial_ax.set_xticks(list(positions))
    financial_ax.set_xticklabels(quarterly.index.strftime("%Y-%m-%d"), rotation=35, ha="right")
    operating_margin = operating_income.div(revenue.where(revenue > 0)) * 100
    margin_ax = financial_ax.twinx()
    margin_ax.plot(
        list(positions),
        operating_margin,
        color="#c44536",
        marker="o",
        linewidth=1.8,
        label="Operating margin",
    )
    margin_ax.set_ylabel("Operating margin (%)")
    handles, labels = financial_ax.get_legend_handles_labels()
    margin_handles, margin_labels = margin_ax.get_legend_handles_labels()
    financial_ax.legend(handles + margin_handles, labels + margin_labels, fontsize=8, loc="upper left")
else:
    financial_ax.text(0.5, 0.5, "Quarterly financials unavailable", ha="center", va="center", transform=financial_ax.transAxes)

# News panel: daily article counts, stacked by publisher.
news_ax = axes[2]
news_ax.set_title(f"News volume, last {news_days} days\nProvider: {news_provider}; legend shows publishers")
news_ax.set_xlabel("Publication date (UTC)")
news_ax.set_ylabel("Article count")
news_ax.grid(True, axis="y", alpha=0.25)
if not news.empty:
    news["publish_day"] = news["published_at"].dt.strftime("%Y-%m-%d")
    top_publishers = news["publisher"].value_counts().nlargest(4).index
    news["publisher_group"] = news["publisher"].where(news["publisher"].isin(top_publishers), "Other")
    daily_counts = pd.crosstab(news["publish_day"], news["publisher_group"]).sort_index()
    daily_counts.plot(kind="bar", stacked=True, ax=news_ax, colormap="tab20")
    news_ax.set_xticklabels(daily_counts.index, rotation=40, ha="right")
    news_ax.legend(title="Publisher", fontsize=7)
else:
    note = news_note or "No dated articles returned"
    news_ax.text(0.5, 0.5, note, ha="center", va="center", transform=news_ax.transAxes)

fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## 6. Evaluation and Iteration

Scores across tickers/runs (visualization), what changed, limitations.